# 🔑 VERSION ENSEIGNANT — CORRIGÉ
Identique à la version étudiant, avec les cellules « À COMPLÉTER » remplies, un **corrigé** après chaque mission et un **dépannage** à la fin. Le découpage horaire n'existe que dans cette version.
Avant la séance : déposer `abdomen_dynamique.nii.gz` (200 images, 0,3 s entre images) à côté du notebook ou dans `./data/`, puis *Run All* une fois (tout doit être ✅).

# TP — Estimation du mouvement respiratoire sur une IRM dynamique
**Master 2 · Traitement d'images médicales · Durée : 2 h**

## 🎯 Objectifs
1. charger et inspecter une séquence IRM NIfTI ;
2. normaliser les intensités ;
3. construire un filtre gaussien ;
4. extraire des contours avec le gradient ;
5. relier diffusion, Laplacien et régularisation ;
6. reconstruire une zone manquante ;
7. estimer un champ de déplacement entre deux images ;
8. suivre le mouvement dans le temps : **relatif cumulé** ou **référence absolue**.

> ⚠️ L'algorithme de mouvement est une **version pédagogique simplifiée**, pas une méthode validée cliniquement.

**Mode d'emploi** : exécutez les cellules dans l'ordre et remplacez les `...` des lignes `À COMPLÉTER`. Les cellules ✅/❌ vérifient vos réponses. Une erreur `'ellipsis'` = il reste un `...`.

## 🗺️ Découpage horaire (enseignant)
| Mission | Thème | Durée |
|---|---|---|
| 1 | Charger et inspecter | 0 – 8 min |
| 2 | Normalisation | 8 – 15 min |
| 3 | Filtre gaussien | 15 – 23 min |
| 4 | Gradient et contours | 23 – 30 min |
| 5 | Diffusion et régularisation | 30 – 42 min |
| 6 | Zone manquante | 42 – 52 min |
| 7 | Champ de déplacement | 52 – 85 min |
| 8 | Suivi temporel (relatif cumulé vs référence absolue) | 85 – 115 min |
| — | Bilan | 115 – 120 min |

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from scipy import ndimage
from IPython.display import display, HTML
from matplotlib.animation import FuncAnimation

plt.rcParams.update({"figure.dpi": 100, "image.interpolation": "nearest", "animation.embed_limit": 80})

# --- Paramètres globaux ---------------------------------------------------------
DT_ACQ = 0.3            # intervalle entre deux images (secondes)
N_IMAGES = 200          # nombre d'images attendu
LAM = 0.1               # régularisation du recalage
SIGMA_PRE = 1.0         # lissage avant dérivation (pixels)


def imshow_cb(ax, img, titre="", cmap="gray", vmin=None, vmax=None):
    """imshow avec barre de couleur (pour lire la plage d'intensité)."""
    im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(titre, fontsize=10)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


VALIDATIONS = {}
def valider(nom, condition, aide=""):
    VALIDATIONS[nom] = bool(condition)
    print(("✅ " if condition else "❌ ") + nom + ("" if condition else "  →  " + aide))


def a_faire(**kw):
    for nom, val in kw.items():
        if val is Ellipsis:
            raise NotImplementedError(f"Complétez d'abord « {nom} » (il reste un `...`).")

---
# Mission 1 — Charger et inspecter la séquence (0 – 8 min)
Un fichier NIfTI peut ranger ses axes dans différents ordres : `(x, y, t)`, `(t, y, x)`… **Inspectez les dimensions avant tout traitement**, vérifiez vos hypothèses (3D, un seul axe de 200 images, pas de NaN) et identifiez l'axe temporel : ne le choisissez pas au hasard. Nous travaillerons avec la convention **(temps, hauteur, largeur)**.

> 💡 NIfTI range les axes dans l'ordre `(i, j, …)` : l'image peut apparaître **couchée**. Dans ce cas, passez `TRANSPOSER = True` dans la cellule ci-dessous.

In [ ]:
FICHIER = next((c for c in ["abdomen_dynamique.nii.gz", "data/abdomen_dynamique.nii.gz", "../data/abdomen_dynamique.nii.gz"]
                if os.path.exists(c)), None)
if FICHIER is None:
    raise FileNotFoundError(f"Fichier introuvable (dossier courant : {os.getcwd()}). Placez-le à côté du notebook ou dans ./data/")

img_nii = nib.load(FICHIER)
vol = np.squeeze(img_nii.get_fdata(dtype=np.float32))      # on retire les axes de taille 1

print("Dimensions :", vol.shape, "| type :", vol.dtype)
print("Pas de temps de l'en-tête (peut être faux) :", img_nii.header.get_zooms())


def verifier_hypotheses(v):
    valider("H1 : données 3D (2 axes spatiaux + 1 axe temporel)", v.ndim == 3,
            f"v.ndim = {v.ndim} : plusieurs coupes ou dimension en trop")
    cand = [i for i, n in enumerate(v.shape) if n == N_IMAGES]
    valider(f"H2 : un seul axe de longueur {N_IMAGES}", len(cand) == 1, f"axes de longueur {N_IMAGES} : {cand}")
    valider("H3 : valeurs finies (pas de NaN)", np.isfinite(v).all(), "NaN ou infinis")
    print("Axes candidats pour le temps :", cand)
    return cand


candidats = verifier_hypotheses(vol)

In [ ]:
# À COMPLÉTER : numéro de l'axe temporel (0, 1 ou 2), d'après vol.shape et la liste des candidats
time_axis = candidats[0]
a_faire(time_axis=time_axis)

seq = np.moveaxis(vol, time_axis, 0)          # (temps, hauteur, largeur)
TRANSPOSER = False                            # passez à True si l'image apparaît « couchée »
if TRANSPOSER:
    seq = np.transpose(seq, (0, 2, 1))
TEMPS = np.arange(len(seq)) * DT_ACQ          # axe des temps en secondes
print("Séquence :", seq.shape, "| durée :", TEMPS[-1] + DT_ACQ, "s")
valider("M1 : 200 images", seq.shape[0] == N_IMAGES, "mauvais axe temporel")

In [ ]:
# Le film (même échelle d'intensité pour toutes les images)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(seq[0], cmap="gray", vmin=seq.min(), vmax=seq.max())
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.axis("off")

def maj(t):
    im.set_data(seq[t])
    ax.set_title(f"t = {TEMPS[t]:.1f} s")

anim = FuncAnimation(fig, maj, frames=range(0, len(seq), 2), interval=150)
plt.close(fig)
display(HTML(anim.to_jshtml()))

### 🔍 Où bouge-t-il ? Carte d'écart-type et kymogramme
- **Carte d'écart-type temporel** : un pixel traversé par un contour qui bouge a un écart-type élevé.
- **Kymogramme** : on prend **une seule colonne** de pixels et on la représente au cours du temps (hauteur en ordonnée, temps en abscisse). Un organe qui oscille verticalement dessine une **sinusoïde** : c'est la respiration que l'on « voit ».

À vous : extraire la colonne `x0` de **toutes** les images, sous forme d'un tableau `(hauteur, temps)`.
> 💡 `seq[:, :, x0]` a la forme `(temps, hauteur)` : il faut le **transposer** (`.T`).

In [ ]:
H, W = seq.shape[1:]
x0 = W // 2
carte_std = seq.std(axis=0)

# À COMPLÉTER : colonne x0 de toutes les images, mise sous forme (hauteur, temps)
kymo = seq[:, :, x0].T
a_faire(kymo=kymo)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
imshow_cb(ax[0], carte_std, "Écart-type temporel par pixel", cmap="magma")
ax[0].axvline(x0, color="cyan", lw=1)
im = ax[1].imshow(kymo, cmap="gray", aspect="auto", extent=[0, len(seq) * DT_ACQ, H, 0])
ax[1].set_title(f"Kymogramme (colonne x = {x0})", fontsize=10)
ax[1].set_xlabel("temps (s)")
ax[1].set_ylabel("hauteur (pixel)")
plt.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04)
ax[2].plot(TEMPS, seq.mean(axis=(1, 2)))
ax[2].set_title("Intensité moyenne de l'image", fontsize=10)
ax[2].set_xlabel("temps (s)")
plt.tight_layout()
valider("M1 : kymogramme de forme (hauteur, temps)", kymo.shape == (H, len(seq)), "transposez : seq[:, :, x0].T")

**Questions** : (1) Quelle région bouge le plus ? Le mouvement est-il uniforme ? (2) Que montre la barre de couleur sur la plage des intensités ? (3) Sur le kymogramme, quelle est la période respiratoire approximative **en secondes** ?

**✍️ Vos réponses :**

### 🔑 Corrigé — Mission 1
- `time_axis` : l'unique axe de longueur 200 (souvent 2). Si `vol.ndim == 4` avec plusieurs coupes : fixer une coupe avant `squeeze`. Si l'image apparaît couchée : `seq = np.transpose(seq, (0, 2, 1))`.
- Q1 : surtout le diaphragme / dôme hépatique ; mouvement **non uniforme** (quasi nul près de la colonne).
- Q2 : les intensités IRM ne sont pas limitées à 0–255 (échelle arbitraire). Q3 : lire la période sur la sinusoïde du kymogramme (nombre d'images par cycle × `DT_ACQ`) ; une respiration normale dure ≈ 3–5 s.
- Kymogramme : `kymo = seq[:, :, x0].T`. Hypothèses H1/H2 en échec ⇒ voir le Dépannage (axe de 200 images en double, plusieurs coupes).

---
# Mission 2 — Normaliser les intensités (8 – 15 min)
Les intensités IRM n'ont pas d'échelle absolue. On les ramène entre 0 et 1 avec les **percentiles 1 et 99** (robustes aux valeurs extrêmes) :

$$\tilde I=\operatorname{clip}\!\left(\frac{I-p_1}{p_{99}-p_1+10^{-8}},\,0,\,1\right)$$

### 👀 Histogramme interactif
Faites varier l'image (`t`) et les **percentiles** bas / haut : l'histogramme (traits rouge et vert = bornes $p_1$ et $p_{99}$), l'image normalisée et la barre de couleur se mettent à jour. Observez ce que l'on **écrête** et ce que l'on gagne en contraste.
> Si les curseurs n'apparaissent pas (`ipywidgets` absent), la cellule affiche simplement le cas par défaut.

In [ ]:
def montrer_hist(t=0, p_bas=1.0, p_haut=99.0):
    img = seq[t].astype(float)
    lo, hi = np.percentile(img, [p_bas, p_haut])
    img01 = np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    imshow_cb(ax[0], img, f"Brute (image {t})")
    imshow_cb(ax[1], img01, "Normalisée", vmin=0, vmax=1)
    ax[2].hist(img.ravel(), bins=80, color="gray")
    ax[2].axvline(lo, color="red", ls="--", label=f"p{p_bas:g} = {lo:.0f}")
    ax[2].axvline(hi, color="green", ls="--", label=f"p{p_haut:g} = {hi:.0f}")
    ax[2].set_title("Histogramme des intensités brutes", fontsize=10)
    ax[2].set_xlabel("intensité")
    ax[2].set_ylabel("nombre de pixels")
    ax[2].legend()
    plt.tight_layout()


try:
    import ipywidgets as widgets
    widgets.interact(montrer_hist,
                     t=widgets.IntSlider(value=0, min=0, max=len(seq) - 1, description="image"),
                     p_bas=widgets.FloatSlider(value=1.0, min=0, max=20, step=0.5, description="p bas"),
                     p_haut=widgets.FloatSlider(value=99.0, min=80, max=100, step=0.5, description="p haut"))
except ImportError:
    montrer_hist()

In [ ]:
def normaliser(image):
    lo, hi = np.percentile(image, [1, 99])
    # À COMPLÉTER : ramener les valeurs entre 0 et 1 (formule ci-dessus, avec np.clip)
    image01 = np.clip((image - lo) / (hi - lo + 1e-8), 0, 1)
    return image01

seq01 = np.stack([normaliser(f.astype(float)) for f in seq])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
imshow_cb(ax[0], seq[0], "Brute")
imshow_cb(ax[1], seq01[0], "Normalisée", vmin=0, vmax=1)
plt.tight_layout()
valider("M2 : valeurs dans [0, 1]", seq01.min() >= 0 and seq01.max() <= 1, "pensez à np.clip(..., 0, 1)")
valider("M2 : le maximum atteint 1", np.isclose(seq01[0].max(), 1.0), "(image - lo) / (hi - lo + 1e-8)")

**Questions** : (1) La normalisation crée-t-elle de l'information ? (2) Pourquoi les percentiles plutôt que min/max ? À quoi sert `np.clip` ?

**✍️ Vos réponses :**

### 🔑 Corrigé — Mission 2
- `image01 = np.clip((image - lo) / (hi - lo + 1e-8), 0, 1)`.
- Q1 : non (transformation monotone), elle en **détruit** même un peu (saturation des 2 % extrêmes). Q2 : min/max dépendent d'un seul pixel aberrant ; `np.clip` évite les valeurs hors de [0, 1] (l'information au-delà de $p_{99}$ est perdue).

---
# Mission 3 — Le filtre gaussien (15 – 23 min)
Il remplace chaque pixel par une **moyenne pondérée** de ses voisins, avec des poids gaussiens :

$$G_\sigma(x)\propto e^{-x^2/(2\sigma^2)}$$

- $\sigma$ (pixels) règle la largeur du lissage : plus il est grand, plus le **bruit** est réduit mais plus les **contours** sont flous ;
- le noyau est **normalisé** (somme = 1) et **séparable** : on filtre les lignes, puis les colonnes ;
- c'est un filtre **passe-bas** (il atténue bruit et détails fins).

In [ ]:
def noyau_gaussien(sigma):
    rayon = int(4 * sigma + 0.5)
    x = np.arange(-rayon, rayon + 1)
    # À COMPLÉTER : exp(-x² / (2σ²))
    g = np.exp(-x ** 2 / (2 * sigma ** 2))
    a_faire(g=g)
    # À COMPLÉTER : normaliser pour que la somme vaille 1
    g = g / g.sum()
    return g

def filtre_gaussien(img, sigma):
    g = noyau_gaussien(sigma)
    return ndimage.convolve1d(ndimage.convolve1d(img, g, axis=0), g, axis=1)     # lignes puis colonnes

img01 = seq01[0]
valider("M3 : noyau normalisé", np.isclose(noyau_gaussien(2).sum(), 1.0), "g = g / g.sum()")
valider("M3 : identique à ndimage.gaussian_filter", np.allclose(filtre_gaussien(img01, 2), ndimage.gaussian_filter(img01, 2), atol=1e-6))

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
imshow_cb(ax[0], img01, "Originale", vmin=0, vmax=1)
for a, s in zip(ax[1:], [1, 2, 4]):
    imshow_cb(a, filtre_gaussien(img01, s), f"σ = {s}", vmin=0, vmax=1)
plt.tight_layout()

**Questions** : (1) Pourquoi normaliser le noyau ? (2) Que deviennent bruit et contours quand $\sigma$ augmente ? Comment choisir $\sigma$ ?

**✍️ Vos réponses :**

### 🔑 Corrigé — Mission 3
- `g = np.exp(-x**2 / (2*sigma**2))` ; `g = g / g.sum()`.
- Q1 : sans normalisation, la luminosité moyenne serait multipliée par la somme du noyau. Q2 : le bruit diminue (≈ $1/(2\sigma\sqrt\pi)$ pour un bruit blanc), les contours s'étalent ; on choisit le plus petit $\sigma$ qui rend le bruit acceptable pour la suite.
- Séparable : $2k$ multiplications par pixel au lieu de $k^2$.

---
# Mission 4 — Gradient et contours (23 – 30 min)
Le gradient mesure les variations locales d'intensité : $|\nabla I|=\sqrt{I_x^2+I_y^2}$. Il est grand sur les **contours**, mais aussi sur le **bruit** : on lisse donc d'abord.

> ⚠️ `np.gradient(img)` renvoie d'abord la dérivée suivant les **lignes** (`gy`, verticale), puis suivant les colonnes (`gx`).

In [ ]:
img01 = seq01[0]

# À COMPLÉTER : dérivées (lignes puis colonnes), puis norme du gradient
gy, gx = np.gradient(img01)
edges = np.sqrt(gx ** 2 + gy ** 2)
a_faire(gy=gy, gx=gx, edges=edges)

# À COMPLÉTER : lisser img01 (gaussien, sigma = 1.5) puis recalculer la norme du gradient
img_lisse = ndimage.gaussian_filter(img01, 1.5)
a_faire(img_lisse=img_lisse)
gy_s, gx_s = np.gradient(img_lisse)
edges_s = np.sqrt(gx_s ** 2 + gy_s ** 2)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
imshow_cb(ax[0], img01, "Image", vmin=0, vmax=1)
imshow_cb(ax[1], edges, "|∇I| sans lissage", cmap="magma", vmin=0, vmax=np.percentile(edges, 99.5))
imshow_cb(ax[2], edges_s, "|∇I| après lissage", cmap="magma", vmin=0, vmax=np.percentile(edges, 99.5))
plt.tight_layout()
valider("M4 : norme du gradient correcte", np.allclose(edges, np.hypot(*np.gradient(img01)[::-1])), "edges = sqrt(gx² + gy²), avec gy, gx = np.gradient(img)")
valider("M4 : le lissage réduit le gradient moyen", edges_s.mean() < edges.mean(), "img_lisse = ndimage.gaussian_filter(img01, 1.5)")

**Questions** : (1) Pourquoi le bruit produit-il des « contours » ? (2) Pourquoi le gradient est-il élevé le long du diaphragme ?

**✍️ Vos réponses :**

### 🔑 Corrigé — Mission 4
- `gy, gx = np.gradient(img01)` ; `edges = np.sqrt(gx**2 + gy**2)` ; `img_lisse = ndimage.gaussian_filter(img01, 1.5)`.
- Piège : écrire `gx, gy = np.gradient(...)` (l'auto-vérification échoue).
- Q1 : une dérivée **amplifie les hautes fréquences**, donc le bruit ; on lisse avant (passe-bas). Q2 : saut d'intensité marqué entre le foie (clair) et le poumon (sombre).

---
# Mission 5 — Diffusion et régularisation (30 – 42 min)
Le **Laplacien** $\Delta u$ = (somme des 4 voisins) − 4 × (pixel) : il vaut 0 sur une zone plate. La mise à jour

$$u\leftarrow u+\alpha\,\Delta u$$

« tire » chaque pixel vers la moyenne de ses voisins : c'est la **diffusion** (elle équivaut à un filtre gaussien).

On mesure la rugosité d'une image par l'**énergie de régularité** (norme $L^2$ du gradient) :

$$R(u)=\tfrac12\sum|\nabla u|^2 .$$

Sa dérivée est $-\Delta u$ : **diffuser = faire une descente de gradient sur $R$**. C'est ce terme qui régularisera le champ de déplacement (Mission 7). Un schéma itératif correct fait **décroître** l'énergie : on la trace à chaque fois.

In [ ]:
def energie_regularite(u):
    gy_, gx_ = np.gradient(u)
    return 0.5 * np.sum(gx_ ** 2 + gy_ ** 2)

In [ ]:
img01 = seq01[0]
u_diff = img01.copy()
alpha, n_iter_diff = 0.15, 30
energies = [energie_regularite(u_diff)]

for k in range(n_iter_diff):
    # À COMPLÉTER : Laplacien de u_diff, puis mise à jour u <- u + alpha * Laplacien
    lap = ndimage.laplace(u_diff)
    u_diff = u_diff + alpha * lap
    energies.append(energie_regularite(u_diff))
a_faire(u_diff=u_diff)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
imshow_cb(ax[0], img01, "Originale", vmin=0, vmax=1)
imshow_cb(ax[1], u_diff, f"Après {n_iter_diff} diffusions", vmin=0, vmax=1)
ax[2].semilogy(energies, marker=".")
ax[2].set_title("Énergie R(u) par itération", fontsize=10)
ax[2].set_xlabel("itération")
ax[2].set_ylabel("R(u)")
plt.tight_layout()
valider("M5 : R(u) décroît à chaque itération", np.all(np.diff(energies) <= 0), "vérifiez le signe : u + alpha * lap")
valider("M5 : l'image est lissée", energies[-1] < 0.8 * energies[0], "lap = ndimage.laplace(u_diff)")

💬 **Lecture** : l'énergie chute vite au début (le bruit disparaît en premier) puis ralentit ; elle ne s'annule qu'à l'infini (image constante). Le schéma n'est stable que pour $\alpha\le 1/4$ : au-delà, l'énergie **explose**.

### Pour aller plus loin : Perona-Malik (lecture)
La diffusion linéaire floute **aussi les contours**. Perona-Malik diffuse **beaucoup dans les zones homogènes** et **peu à travers les contours**, grâce à une conductivité qui dépend du gradient :
$\partial_t u=\operatorname{div}\big(g(|\nabla u|)\nabla u\big)$, $g(s)=e^{-(s/\kappa)^2}$ ($\kappa$ = seuil bruit / contour). C'est la descente de gradient d'une pénalité qui **sature** pour les grands gradients (au lieu de la pénalité quadratique $L^2$) : les contours sont « tolérés ». Défaut : effet d'escalier, choix de $\kappa$.

In [ ]:
def perona_malik(u0, n_iter=30, kappa=0.08, gamma=0.2):
    u = u0.copy()
    g = lambda d: np.exp(-(d / kappa) ** 2)
    for _ in range(n_iter):
        p = np.pad(u, 1, mode="edge")
        dN, dS, dW, dE = p[:-2, 1:-1] - u, p[2:, 1:-1] - u, p[1:-1, :-2] - u, p[1:-1, 2:] - u
        u = u + gamma * (g(dN) * dN + g(dS) * dS + g(dW) * dW + g(dE) * dE)
    return u

u_lin = img01.copy()
for _ in range(30):
    u_lin = u_lin + 0.2 * ndimage.laplace(u_lin)
u_pm = perona_malik(img01)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
imshow_cb(ax[0], img01, "Originale", vmin=0, vmax=1)
imshow_cb(ax[1], u_lin, "Diffusion linéaire (contours flous)", vmin=0, vmax=1)
imshow_cb(ax[2], u_pm, "Perona-Malik (contours préservés)", vmin=0, vmax=1)
plt.tight_layout()

**Questions** : (1) Pourquoi $\alpha>1/4$ est-il instable ? (2) Comment savoir quand arrêter les itérations ? (3) Pourquoi régularisera-t-on le champ de déplacement par la norme $L^2$ ? Qu'y perd-on ?

**✍️ Vos réponses :**

### 🔑 Corrigé — Mission 5
- `lap = ndimage.laplace(u_diff)` ; `u_diff = u_diff + alpha * lap`. Erreur classique : `-` → anti-diffusion, l'énergie remonte (détectée).
- Q1 : le mode « damier » est multiplié par $|1-8\alpha|$ à chaque itération : > 1 si $\alpha>1/4$. Q2 : quand le gain d'énergie par itération devient négligeable (ou quand $\sigma=\sqrt{2\alpha k}$ atteint le lissage voulu).
- Q3 : $L^2$ = convexe, problème **linéaire** (Laplacien), simple et robuste. On perd les **discontinuités** du mouvement (glissement foie/paroi) : le champ est trop lisse à ces endroits.
- Diffusion linéaire ≡ filtre gaussien de $\sigma=\sqrt{2\alpha k}$.

---
# Mission 6 — Reconstruire une zone manquante (42 – 52 min)
On crée un **trou artificiel** (la vérité est connue, on peut donc mesurer l'erreur). À chaque itération : (1) on lisse l'image, (2) on **copie** les valeurs lissées dans le trou, (3) on **réimpose** les pixels connus (fidélité aux données).

> ⚠️ Méthode illustrative très simple, pas une méthode générale de reconstruction médicale.

In [ ]:
img01 = seq01[0]
H, W = img01.shape
mask = np.zeros((H, W), dtype=bool)                      # True = pixel manquant
mask[H // 3: H // 3 + H // 12, W // 3: W // 3 + W // 12] = True
img_corrupted = img01.copy()
img_corrupted[mask] = 0

In [ ]:
recon = img_corrupted.copy()
erreurs = []                                             # erreur RMSE dans le trou

for k in range(200):
    smooth = ndimage.gaussian_filter(recon, sigma=1)
    # À COMPLÉTER : valeurs lissées dans le trou, valeurs observées ailleurs
    recon[mask] = smooth[mask]
    recon[~mask] = img01[~mask]
    erreurs.append(np.sqrt(np.mean((recon[mask] - img01[mask]) ** 2)))

fig, ax = plt.subplots(1, 4, figsize=(17, 4))
imshow_cb(ax[0], img_corrupted, "Avec trou", vmin=0, vmax=1)
imshow_cb(ax[1], recon, "Reconstruction", vmin=0, vmax=1)
imshow_cb(ax[2], img01, "Vérité", vmin=0, vmax=1)
ax[3].semilogy(erreurs)
ax[3].set_title("Erreur RMSE dans le trou", fontsize=10)
ax[3].set_xlabel("itération")
plt.tight_layout()
valider("M6 : pixels connus inchangés", np.allclose(recon[~mask], img01[~mask]), "recon[~mask] = img01[~mask]")
valider("M6 : mieux qu'un trou noir", erreurs[-1] < np.sqrt(np.mean(img01[mask] ** 2)), "recon[mask] = smooth[mask]")

💬 **Lecture** : l'erreur chute puis atteint un **plateau** : on a trouvé la solution « la plus lisse » compatible avec les bords du trou, mais la texture n'est pas retrouvée. C'est un **problème inverse** : $\min_u\|Mu-y\|^2+\lambda R(u)$ (fidélité aux données + régularité).

**Questions** : (1) Pourquoi réimposer les pixels connus ? (2) Si le trou est beaucoup plus grand, peut-on retrouver l'anatomie disparue ? Pourquoi ?

**✍️ Vos réponses :**

### 🔑 Corrigé — Mission 6
- `recon[mask] = smooth[mask]` ; `recon[~mask] = img01[~mask]`.
- Q1 : sans cela le lissage s'applique aussi aux pixels connus (image floue, on perd la fidélité aux données). Q2 : non : problème **mal posé**, une infinité d'images sont compatibles ; la régularité en choisit une (la plus lisse), pas la vraie. Plus le trou est grand, plus le plateau d'erreur est haut ; une méthode apprise peut « halluciner » des structures plausibles mais fausses.

---
# Mission 7 — Estimer un champ de déplacement (52 – 85 min)
On cherche, entre deux images $I_0$ et $I_1$, un déplacement horizontal $u(x,y)$ et vertical $v(x,y)$ (en pixels). Si l'intensité est conservée et le déplacement petit :

$$r=I_x\,u+I_y\,v+I_t,\qquad I_t=I_1-I_0 .$$

On minimise un **terme de données** + une **régularisation** (celle de la Mission 5) :

$$E(u,v)=\tfrac12\sum r^2+\tfrac{\lambda}{2}\sum\big(|\nabla u|^2+|\nabla v|^2\big).$$

Son gradient est $I_x r-\lambda\Delta u$ (et $I_y r-\lambda\Delta v$) : on fait une **descente de gradient** $u\leftarrow u-\eta\,(I_x r-\lambda\Delta u)$, $v\leftarrow v-\eta\,(I_y r-\lambda\Delta v)$, avec un pas $\eta$ calculé automatiquement.

**Limites** : petits déplacements seulement ; intensité supposée constante ; dans une zone uniforme ($I_x=I_y=0$) seule la régularisation fixe le champ. Ici $\lambda=$ `LAM` $=0{,}1$.

In [ ]:
def preparer_paire(I0, I1):
    """Lissage léger puis dérivées Ix, Iy et différence temporelle It."""
    A, B = ndimage.gaussian_filter(I0, SIGMA_PRE), ndimage.gaussian_filter(I1, SIGMA_PRE)
    gyA, gxA = np.gradient(A)
    gyB, gxB = np.gradient(B)
    return 0.5 * (gxA + gxB), 0.5 * (gyA + gyB), B - A


def energie(u, v, Ix, Iy, It, lam):
    r = Ix * u + Iy * v + It
    guy, gux = np.gradient(u)
    gvy, gvx = np.gradient(v)
    return 0.5 * np.sum(r ** 2) + 0.5 * lam * np.sum(gux ** 2 + guy ** 2 + gvx ** 2 + gvy ** 2)


def warper(I1, u, v):
    """I1 déformée : I1(x+u, y+v). Si le champ est bon, elle ressemble à I0."""
    Y, X = np.mgrid[0:I1.shape[0], 0:I1.shape[1]].astype(float)
    return ndimage.map_coordinates(I1, [Y + v, X + u], order=1, mode="nearest")

    return u, v

### 1️⃣ Choisir deux images
On prend `I0` à l'instant où le mouvement est le plus rapide, et `I1` deux images plus tard (`ECART = 2`, soit 0,6 s). Les images ne sont **pas forcément consécutives** : plus elles sont éloignées, plus le déplacement est grand et moins l'approximation linéaire est valable.

In [ ]:
ECART = 2
activite = np.abs(seq01[ECART:] - seq01[:-ECART]).mean(axis=(1, 2))
T0 = int(np.argmax(activite))
I0, I1 = seq01[T0].astype(float), seq01[T0 + ECART].astype(float)
Ix, Iy, It = preparer_paire(I0, I1)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
imshow_cb(ax[0], I0, f"I0 : image {T0} (t = {TEMPS[T0]:.1f} s)", vmin=0, vmax=1)
imshow_cb(ax[1], I1, f"I1 : image {T0 + ECART}", vmin=0, vmax=1)
m = np.abs(It).max()
imshow_cb(ax[2], It, "It = I1 − I0", cmap="coolwarm", vmin=-m, vmax=m)
plt.tight_layout()

### 2️⃣ ✍️ À vous : la descente de gradient
Complétez les 5 lignes de la boucle (résidu, deux Laplaciens, deux mises à jour).

In [ ]:
def estimer_gd(Ix, Iy, It, lam=LAM, n_iter=300):
    eta = 0.9 / ((Ix ** 2 + Iy ** 2).max() + 8 * lam)          # pas stable (calculé automatiquement)
    u, v = np.zeros_like(Ix), np.zeros_like(Ix)
    energies = []
    for k in range(n_iter):
        # À COMPLÉTER : résidu, Laplaciens de u et de v, mises à jour
        r = Ix * u + Iy * v + It
        lap_u = ndimage.laplace(u)
        lap_v = ndimage.laplace(v)
        u = u - eta * (Ix * r - lam * lap_u)
        v = v - eta * (Iy * r - lam * lap_v)
        energies.append(energie(u, v, Ix, Iy, It, lam))
    return u, v, energies

u, v, energies = estimer_gd(Ix, Iy, It)
print("Énergie : %.2f -> %.2f | |v| max = %.2f px" % (energies[0], energies[-1], np.abs(v).max()))
valider("M7 : l'énergie décroît", np.all(np.diff(energies) <= 1e-9 * energies[0]), "erreur de signe dans la mise à jour ?")
valider("M7 : le champ n'est pas nul", np.abs(v).max() > 1e-3, "r, lap_u, lap_v ou les mises à jour sont mal écrits")

### 3️⃣ Visualiser le résultat
Les flèches sont **agrandies** (`GROSSISSEMENT = 5`) car les déplacements sont de l'ordre du pixel. Le **résidu après déformation** doit être plus faible qu'avant.

In [ ]:
GROSSISSEMENT = 5
step = 8
Y, X = np.mgrid[0:I0.shape[0]:step, 0:I0.shape[1]:step]
I1w = warper(I1, u, v)
vm = np.percentile(np.abs(I1 - I0), 99.5)

fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))
ax[0].imshow(I0, cmap="gray", vmin=0, vmax=1)
q = ax[0].quiver(X, Y, u[::step, ::step], v[::step, ::step], np.hypot(u, v)[::step, ::step],
                 cmap="autumn", angles="xy", scale_units="xy", scale=1 / GROSSISSEMENT)
plt.colorbar(q, ax=ax[0], fraction=0.046, pad=0.04, label="pixels")
ax[0].set_title(f"Déplacement estimé (flèches ×{GROSSISSEMENT})", fontsize=10)
ax[0].axis("off")
mm = max(np.abs(v).max(), 1e-3)
imshow_cb(ax[1], v, "Déplacement vertical v (pixels)", cmap="coolwarm", vmin=-mm, vmax=mm)
imshow_cb(ax[2], np.abs(I1w - I0), f"|I1 déformée − I0| (RMSE {np.sqrt(np.mean((I1w - I0) ** 2)):.4f})", cmap="magma", vmin=0, vmax=vm)
ax[3].semilogy(energies)
ax[3].set_title("Énergie E par itération", fontsize=10)
ax[3].set_xlabel("itération")
plt.tight_layout()
print("RMSE avant déformation : %.4f" % np.sqrt(np.mean((I1 - I0) ** 2)))

💬 **Lecture** : l'énergie décroît mais **lentement** (la descente de gradient converge mal dans les zones sans contraste) ; le résidu diminue sans s'annuler. Le champ est concentré sur les **contours** : à l'intérieur d'un organe uniforme, rien ne contraint le mouvement (problème d'ouverture).

### 4️⃣ Effet de $\lambda$

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, lm in zip(ax, [0.0, LAM, 1.0]):
    uu, vv, _ = estimer_gd(Ix, Iy, It, lam=lm)
    rm = np.sqrt(np.mean((warper(I1, uu, vv) - I0) ** 2))
    mm = max(np.abs(vv).max(), 1e-3)
    imshow_cb(a, vv, f"v pour λ = {lm}  (RMSE résiduelle {rm:.4f})", cmap="coolwarm", vmin=-mm, vmax=mm)
plt.tight_layout()

**Questions** : (1) Quel terme utilise l'information des images ? Lequel favorise un champ lisse ? (2) Que fait $\lambda$ quand il augmente ? (3) Pourquoi un champ plus lisse n'est-il pas forcément plus exact ?

**✍️ Vos réponses :**

### 🔑 Corrigé — Mission 7
```python
r = Ix*u + Iy*v + It
lap_u = ndimage.laplace(u)
lap_v = ndimage.laplace(v)
u = u - eta * (Ix*r - lam*lap_u)
v = v - eta * (Iy*r - lam*lap_v)
```
- Erreur de signe fréquente : `+ lam*lap_u` ou `u + eta*(...)` → l'énergie remonte (détectée).
- Q1 : données = $\frac12\sum r^2$ ; lissage = $\frac\lambda2\sum(|\nabla u|^2+|\nabla v|^2)$. Q2 : $\lambda$ ↑ ⇒ champ plus lisse mais amplitude **sous-estimée** ($\lambda=0$ : champ bruité ; $\lambda=1$ : mouvement écrasé) ; $\lambda$ se compare à $I_x^2+I_y^2$, donc dépend de l'échelle des intensités (ici [0, 1]). Q3 : le vrai champ n'est pas lisse partout (glissement foie/paroi) ; la régularité est une **hypothèse a priori**.
- Attendu : énergie décroissante mais pas convergée à 300 itérations (normal) ; amplitude < 1 px ; résidu plus faible qu'avant. Si `|v|` explose : `eta` mis à la main trop grand.

---
# Mission 8 — Suivre le mouvement dans le temps (85 – 115 min)
On suit la **moyenne du déplacement vertical $v$ dans une région d'intérêt (ROI)** ; c'est un indicateur pédagogique, pas une mesure clinique. Deux façons d'obtenir une courbe de position :

| | **A. Relatif cumulé** | **B. Référence absolue** |
|---|---|---|
| Principe | mouvement entre **images consécutives**, puis **somme cumulée** | mouvement entre **une image de référence fixe** et **chaque image** |
| Déplacement à estimer | petit | grand |
| Erreurs | **s'accumulent** (dérive ?) | **ne s'accumulent pas** |
| Dépend de | rien | du choix de la référence |

Ce sont des hypothèses *a priori* : vous allez les **tester**. On réutilise la même méthode d'estimation qu'en Mission 7 (descente de gradient, 100 itérations par estimation pour limiter le temps de calcul).

In [ ]:
def estimer_mouvement(I0, I1, n_iter=100):
    """Même méthode qu'en Mission 7 (descente de gradient), avec moins d'itérations."""
    Ix, Iy, It = preparer_paire(I0, I1)
    u, v, _ = estimer_gd(Ix, Iy, It, n_iter=n_iter)
    return u, v

H, W = seq01[0].shape
roi = np.zeros((H, W), dtype=bool)
roi[H // 4:H // 2, W // 4:3 * W // 4] = True          # À ADAPTER à la zone qui bouge

plt.figure(figsize=(5, 4))
plt.imshow(seq01[0], cmap="gray", vmin=0, vmax=1)
plt.contour(roi, levels=[0.5], colors="red")
plt.title("Région d'intérêt (rouge)")
plt.axis("off");

### 1️⃣ ✍️ Approche A : mouvement relatif cumulé
Pour chaque paire consécutive, estimez $(u,v)$ et gardez la **moyenne de $v$ dans la ROI** (c'est une vitesse en pixels par pas). Puis **additionnez** pour obtenir la position.

In [ ]:
signal_rel, temps_rel = [], []
for t in range(len(seq01) - 1):
    debut = time.perf_counter()
    u, v = estimer_mouvement(seq01[t].astype(float), seq01[t + 1].astype(float))
    temps_rel.append(time.perf_counter() - debut)
    # À COMPLÉTER : moyenne de v dans la ROI
    signal_rel.append(np.mean(v[roi]))
a_faire(signal=signal_rel[0])

# À COMPLÉTER : position = somme cumulée des déplacements
pos_rel = np.cumsum(signal_rel)
a_faire(pos_rel=pos_rel)
temps_rel = np.array(temps_rel)       # l'élément k correspond à l'image k+1
valider("M8-A : position = somme cumulée", np.allclose(pos_rel, np.cumsum(signal_rel)), "pos_rel = np.cumsum(signal_rel)")
valider("M8-A : le signal oscille", np.sum(np.diff(np.sign(signal_rel)) != 0) >= 2, "adaptez la ROI")

### 2️⃣ ✍️ Approche B : référence absolue
Cette fois, chaque image est comparée à **la même image de référence** (`T_REF = 0`) : la moyenne de $v$ dans la ROI donne **directement** la position, sans somme.

In [ ]:
T_REF = 0
I_ref = seq01[T_REF].astype(float)
signal_abs, temps_abs = [], []
for t in range(1, len(seq01)):
    debut = time.perf_counter()
    # À COMPLÉTER : déplacement entre l'image de référence I_ref et l'image courante
    u, v = estimer_mouvement(I_ref, seq01[t].astype(float))
    temps_abs.append(time.perf_counter() - debut)
    # À COMPLÉTER : moyenne de v dans la ROI
    signal_abs.append(np.mean(v[roi]))
a_faire(signal=signal_abs[0])
signal_abs, temps_abs = np.array(signal_abs), np.array(temps_abs)       # l'élément k correspond à l'image k+1
valider("M8-B : une position par image", len(signal_abs) == len(seq01) - 1)
valider("M8-B : le signal oscille", np.sum(np.diff(np.sign(signal_abs - signal_abs.mean())) != 0) >= 2, "adaptez la ROI")

### 3️⃣ Visualiser A et B
On trace les deux positions et leur **différence** A − B. Comme B n'accumule pas d'erreur, une différence qui **croît** au cours du temps révèle une **dérive** de A.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(TEMPS[1:], pos_rel, label="A : relatif cumulé")
ax[0].plot(TEMPS[1:], signal_abs, label="B : référence absolue")
ax[0].set_title("Position verticale de la ROI (pixels)", fontsize=10)
ax[0].set_xlabel("temps (s)")
ax[0].legend()
ax[1].plot(TEMPS[1:], pos_rel - signal_abs)
ax[1].axhline(0, color="gray", lw=0.5)
ax[1].set_title("Différence A − B (pixels)", fontsize=10)
ax[1].set_xlabel("temps (s)")
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
print("Corrélation A/B : %.3f | calcul par estimation : A %.0f ms, B %.0f ms" %
      (np.corrcoef(pos_rel, signal_abs)[0, 1], 1000 * temps_rel.mean(), 1000 * temps_abs.mean()))

### 4️⃣ ⏱️ Temps réel ?
Les images arrivent toutes les `DT_ACQ = 0,3 s` (valeur **fournie**, ne la déduisez pas du nombre d'images). Un calcul plus rapide que cet intervalle est **nécessaire mais pas suffisant** : il faut aussi compter reconstruction, transferts, affichage. On raisonne sur le **pire cas** (95ᵉ percentile).

In [ ]:
for nom, tc in [("A (relatif)", temps_rel), ("B (référence)", temps_abs)]:
    p95 = 1000 * np.percentile(tc, 95)
    print(f"{nom:<14}: moyenne {1000 * tc.mean():5.1f} ms | p95 {p95:5.1f} ms | marge Δt/p95 = {1000 * DT_ACQ / p95:.1f}")

**Questions** : (1) La différence A − B croît-elle au cours du temps ? Que peut-on en conclure sur A ? (2) Quels avantages / inconvénients de chaque approche vos résultats confirment-ils (calcul, dépendance à la référence, amplitude des déplacements à estimer) ? (3) Laquelle pour le suivi en temps réel ? pour une analyse hors ligne ? Proposez une approche hybride.

**✍️ Vos réponses :**

### 🔑 Corrigé — Mission 8
```python
signal_rel.append(np.mean(v[roi]))
pos_rel = np.cumsum(signal_rel)
u, v = estimer_mouvement(I_ref, seq01[t].astype(float))
signal_abs.append(np.mean(v[roi]))
```
- Le signe dépend de la convention d'axes (`v` > 0 = vers le bas).
- **Dérive** : une erreur systématique $\\epsilon$ par pas donne $\\approx n\\epsilon$ après $n$ pas (cumsum) ; B n'accumule rien. Attendu : A − B **s'écarte de 0 de plus en plus** (pente). Si le biais est faible, la dérive reste discrète : conclusion acceptable (elle croît avec la durée).
- **Amplitude** : B compare des images éloignées, donc de grands déplacements pour lesquels l'approximation linéaire est moins valable : amplitude plus **sous-estimée** ; A estime de petits déplacements. Les deux courbes sont corrélées mais pas identiques.
- **Synthèse** — A : simple, rapide, temps réel ; dérive. B : pas de dérive, repère fixe ; grands déplacements, dépend de la référence, apparence qui change.
- **Hybride** : suivi en A avec **ré-ancrage** périodique sur la référence (même phase du cycle).
- **Temps réel** : à 300 ms le calcul laisse de la marge, mais le budget doit inclure reconstruction, transfert, affichage ; raisonner sur le p95.

**Bilan (éléments de réponse)** : normalisation = échelle robuste sans créer d'information ; gradient = variations locales (contours + bruit) ; Laplacien = dérivée de la régularité $\frac12\|\nabla u\|^2$ (diffuser = descendre son gradient) ; reconstruction = données connues + régularité ; mouvement = terme de données (résidu) + régularisation ; temps réel = calcul au pire cas + budget total comparés à $\Delta t_{\mathrm{acq}}$.

In [ ]:
ok = [k for k, val in VALIDATIONS.items() if val]
ko = [k for k, val in VALIDATIONS.items() if not val]
print(f"BILAN DES AUTO-VÉRIFICATIONS : ✅ {len(ok)} réussies, ❌ {len(ko)} à revoir (sur {len(VALIDATIONS)})")
for k in ko:
    print("   ❌", k)
if not ko:
    print("🎉 Bravo : toutes les vérifications sont passées !")

> **À retenir** : intensités → filtre gaussien → gradients → diffusion et régularisation → champ de déplacement → mouvement relatif cumulé *vs* référence absolue → budget de calcul.

---
# 🛠️ Dépannage (enseignant)
| Symptôme | Cause | Remède |
|---|---|---|
| `FileNotFoundError` | mauvais dossier | placer `abdomen_dynamique.nii.gz` à côté du notebook ou dans `./data/` |
| `assert vol.ndim == 3` / pas de 200 images | plusieurs coupes ou axe en trop | `vol = vol[..., k, :]` avant `squeeze` ; fournir `time_axis` |
| Image couchée | convention NIfTI | `seq = np.transpose(seq, (0, 2, 1))` (puis `u`/`v` suivent les nouveaux axes) |
| `'ellipsis'` / `Complétez d'abord…` | `...` laissé | remplacer par le code demandé |
| Animation absente | backend sans JavaScript | utiliser Jupyter/JupyterLab ou `plt.imshow` |
| M4 ❌ ordre des dérivées | `gx, gy = np.gradient(...)` | `gy, gx = np.gradient(...)` |
| M5 énergie qui remonte | signe `-` ou α > 0,25 | `u + alpha*lap`, α ≤ 0,25 |
| M7 énergie qui remonte / NaN | signe ou `eta` à la main | formules ; garder `eta` automatique |
| M7 champ nul / très faible | λ trop grand, images non lissées | `LAM = 0.1`, `SIGMA_PRE = 1` |
| M8 signal plat | ROI hors zone mobile | repositionner la ROI (zone du diaphragme) |
| M8 boucle lente | 2 × 199 estimations (~30–60 s) | tester sur `range(1, 20)` ; réduire `n_iter` |

**Limites de l'algorithme (à rappeler en conclusion)** : petits déplacements, intensité constante, problème d'ouverture, sous-estimation de l'amplitude, dérive (A), dépendance à la référence (B), pénalité $L^2$ qui lisse les discontinuités, 2D seulement, aucune validation clinique.